# 03. Wavelet-ознаки

**Етап 03 з 13** · NASA C-MAPSS FD001 · RUL prediction


## 1. Дані

In [ ]:
import mlflow

NGROK_URL = "https://stage-broom-alkalize.ngrok-free.dev"
os.environ["MLFLOW_TRACKING_SERVER_CERT_VERIFY"] = "false"
os.environ["HTTP_HEADER_NGROK_SKIP_BROWSER_WARNING"] = "true"

mlflow.set_tracking_uri(NGROK_URL)
mlflow.set_experiment('predictive_maintenance_colab')


In [ ]:
import os
from pathlib import Path

import numpy as np
import pandas as pd
import pywt

from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import GroupShuffleSplit
from xgboost import XGBRegressor

DATA_DIR = Path(os.environ.get("FD001_DIR", "/content/drive/MyDrive/predictive_maintenance"))
SEED = 42

COLUMNS = ["unit_id", "cycle", "setting_1", "setting_2", "setting_3"] + [f"sensor_{i}" for i in range(1, 22)]

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [6]:
def load_train(data_dir):
    """Читає train_FD001.txt і додає RUL: скільки циклів лишилось до останнього циклу двигуна."""
    train = pd.read_csv(data_dir / "train_FD001.txt", sep=r"\s+", header=None, names=COLUMNS)
    train["RUL"] = train.groupby("unit_id")["cycle"].transform("max") - train["cycle"]
    return train

train = load_train(DATA_DIR)

# Поділ за двигунами: 80 двигунів на train, 20 на validation
splitter = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=SEED)
train_idx, val_idx = next(splitter.split(train, groups=train["unit_id"]))

train_df = train.iloc[train_idx].copy()
val_df = train.iloc[val_idx].copy()

print("Train engines:", train_df["unit_id"].nunique(), "| rows:", len(train_df))
print("Validation engines:", val_df["unit_id"].nunique(), "| rows:", len(val_df))

Train engines: 80 | rows: 16561
Validation engines: 20 | rows: 4070


## 2. Ознаки

In [7]:
WAVELET_SENSORS = [
    "sensor_4", "sensor_7", "sensor_11", "sensor_12",
    "sensor_15", "sensor_20", "sensor_21",
]
WAVELET_WINDOW = 16
WAVELET_NAME = "db1"
WAVELET_LEVEL = 2
WAVELET_STATS = ["a2_mean", "d1_std", "d1_energy", "d2_std", "d2_energy"]


def make_wavelet_features(df, stats=WAVELET_STATS):
    """Додає причинні wavelet-ознаки `{sensor}_wlt_{stat}` для кожного сенсора.

    Вікно складається з поточного і минулих циклів двигуна; початок історії доповнюється
    першим значенням. `stats` вибирає статистики з WAVELET_STATS; значення для d2_std
    не залежить від того, які статистики запитано.
    """
    result = df.copy()
    feature_frames = []

    for unit_id, group in result.groupby("unit_id", sort=False):
        group = group.sort_values("cycle")
        sensor_values = group[WAVELET_SENSORS].to_numpy(dtype=float)

        features = {
            sensor: {stat: [] for stat in stats}
            for sensor in WAVELET_SENSORS
        }

        for i in range(len(group)):
            for j, sensor in enumerate(WAVELET_SENSORS):
                history = sensor_values[max(0, i - WAVELET_WINDOW + 1): i + 1, j]

                if len(history) < WAVELET_WINDOW:
                    history = np.pad(
                        history,
                        (WAVELET_WINDOW - len(history), 0),
                        mode="edge",
                    )

                cA2, cD2, cD1 = pywt.wavedec(
                    history,
                    wavelet=WAVELET_NAME,
                    level=WAVELET_LEVEL,
                    mode="symmetric",
                )

                values = {
                    "a2_mean": np.mean(cA2),
                    "d1_std": np.std(cD1),
                    "d1_energy": np.mean(cD1 ** 2),
                    "d2_std": np.std(cD2),
                    "d2_energy": np.mean(cD2 ** 2),
                }

                for stat in stats:
                    features[sensor][stat].append(values[stat])

        feature_data = {
            f"{sensor}_wlt_{stat}": features[sensor][stat]
            for sensor in WAVELET_SENSORS
            for stat in stats
        }

        feature_frames.append(pd.DataFrame(feature_data, index=group.index))

    wavelet_df = pd.concat(feature_frames).sort_index()

    return pd.concat([result, wavelet_df], axis=1)


def wavelet_cols(stats):
    """Повертає назви wavelet-колонок для заданих статистик (спершу сенсор, потім статистика)."""
    return [f"{sensor}_wlt_{stat}" for sensor in WAVELET_SENSORS for stat in stats]

In [ ]:
# Базові ознаки в тому самому порядку, що й у попередньому XGBoost baseline.
BASE_COLS = ["cycle", "setting_1", "setting_2", "setting_3"] + [f"sensor_{i}" for i in range(1, 22)]

WLT_ALL_COLS = wavelet_cols(WAVELET_STATS)
WLT_D2_COLS = wavelet_cols(["d2_std"])

FINAL_FEATURES = BASE_COLS + WLT_D2_COLS

train_wlt = make_wavelet_features(train_df)
val_wlt = make_wavelet_features(val_df)

assert len(WLT_ALL_COLS) == 35
assert set(WLT_ALL_COLS).issubset(train_wlt.columns)
assert set(WLT_ALL_COLS).issubset(val_wlt.columns)

y_train = train_df["RUL"]
y_val = val_df["RUL"]

XGB_PARAMS = {
    "n_estimators": 300,
    "learning_rate": 0.012074507989018332,
    "max_depth": 4,
    "min_child_weight": 3.2298041995296414,
    "subsample": 0.7900204897773602,
    "colsample_bytree": 0.9223090698240903,
    "reg_alpha": 0.7459635164536705,
    "reg_lambda": 4.416722489003329,
    "gamma": 0.2695542352154229,
    "objective": "reg:squarederror",
    "random_state": SEED,
    "n_jobs": -1,
    "tree_method": "hist",
}


def regression_metrics(y_true, y_pred):
    """Повертає MAE, RMSE і R2 для прогнозу."""
    return {
        "MAE": mean_absolute_error(y_true, y_pred),
        "RMSE": np.sqrt(mean_squared_error(y_true, y_pred)),
        "R2": r2_score(y_true, y_pred),
    }

## 3. Експерименти

In [9]:
GROUPS = {stat: wavelet_cols([stat]) for stat in WAVELET_STATS}

CONFIGS = {"Baseline": []}

CONFIGS["Baseline + ALL"] = WLT_ALL_COLS

for group, cols in GROUPS.items():
    CONFIGS[f"Baseline + {group}"] = cols

for group, cols in GROUPS.items():
    CONFIGS[f"ALL - {group}"] = [c for c in WLT_ALL_COLS if c not in cols]

for col in WLT_ALL_COLS:
    CONFIGS[f"Single: {col}"] = [col]

print("Configurations:", len(CONFIGS))

Configurations: 47


In [22]:
rows = []

for name, wlt_cols in CONFIGS.items():
    feats = BASE_COLS + wlt_cols

    model = XGBRegressor(**XGB_PARAMS)
    model.fit(train_wlt[feats], y_train)

    pred = np.clip(model.predict(val_wlt[feats]), 0, None)

    metrics = regression_metrics(y_val, pred)

    rows.append({
        "experiment": name,
        "n_features": len(feats),
        **metrics,
    })

    run_name = f"Wavelets | {name}"
    with mlflow.start_run(run_name=run_name):
      mlflow.log_params({
          "experiment_name": name,
          "model_type": model.__class__.__name__,
          "num_features": len(feats),
          "features_list": ", ".join(feats),
          **XGB_PARAMS
      })

      mlflow.log_metrics(metrics)

results = pd.DataFrame(rows)

🏃 View run Wavelets | Baseline at: https://stage-broom-alkalize.ngrok-free.dev/#/experiments/1/runs/aeedb894c4a141c2b8cabd536fe02c36
🧪 View experiment at: https://stage-broom-alkalize.ngrok-free.dev/#/experiments/1
🏃 View run Wavelets | Baseline + ALL at: https://stage-broom-alkalize.ngrok-free.dev/#/experiments/1/runs/34c8919e8bd54cf2a0dcd2072da91298
🧪 View experiment at: https://stage-broom-alkalize.ngrok-free.dev/#/experiments/1
🏃 View run Wavelets | Baseline + a2_mean at: https://stage-broom-alkalize.ngrok-free.dev/#/experiments/1/runs/e5f478659fb24dcf92d3acb782c38940
🧪 View experiment at: https://stage-broom-alkalize.ngrok-free.dev/#/experiments/1
🏃 View run Wavelets | Baseline + d1_std at: https://stage-broom-alkalize.ngrok-free.dev/#/experiments/1/runs/906db03016794021b1b6f2df64a50a89
🧪 View experiment at: https://stage-broom-alkalize.ngrok-free.dev/#/experiments/1
🏃 View run Wavelets | Baseline + d1_energy at: https://stage-broom-alkalize.ngrok-free.dev/#/experiments/1/runs/8c3

MlflowException: API request to endpoint /api/2.0/mlflow/runs/create failed with error code 503 != 200. Response body: '<!DOCTYPE html>
<html class="h-full" lang="en-US" dir="ltr">
  <head>
    <meta charset="utf-8">
    <meta name="viewport" content="width=device-width, initial-scale=1">
    <link rel="preload" href="https://assets.ngrok.com/fonts/euclid-square/EuclidSquare-Regular-WebS.woff" as="font" type="font/woff" crossorigin="anonymous" />
    <link rel="preload" href="https://assets.ngrok.com/fonts/euclid-square/EuclidSquare-RegularItalic-WebS.woff" as="font" type="font/woff" crossorigin="anonymous" />
    <link rel="preload" href="https://assets.ngrok.com/fonts/euclid-square/EuclidSquare-Medium-WebS.woff" as="font" type="font/woff" crossorigin="anonymous" />
    <link rel="preload" href="https://assets.ngrok.com/fonts/euclid-square/EuclidSquare-MediumItalic-WebS.woff" as="font" type="font/woff" crossorigin="anonymous" />
    <link rel="preload" href="https://assets.ngrok.com/f

## 4. Результати

In [ ]:
baseline_mae = results.loc[results["experiment"] == "Baseline", "MAE"].iloc[0]

results["MAE_delta"] = results["MAE"] - baseline_mae
results["MAE_change_%"] = results["MAE_delta"] / baseline_mae * 100

results = results.sort_values("MAE").reset_index(drop=True)
display(results.round(4))

display(results[results["experiment"].isin(["Baseline", "Baseline + d2_std", "Baseline + ALL"])].round(4))

,experiment,n_features,MAE,RMSE,R2,MAE_delta,MAE_change_%
0,Baseline + d2_std,32,22.9140,29.6756,0.7957,-0.1577,-0.6836
1,Single: sensor_12_wlt_a2_mean,26,22.9368,29.5874,0.7969,-0.1348,-0.5844
2,Single: sensor_11_wlt_d2_std,26,22.9624,29.7779,0.7943,-0.1092,-0.4735
3,Single: sensor_15_wlt_d1_energy,26,22.9755,29.7563,0.7946,-0.0962,-0.4170
4,Single: sensor_20_wlt_d1_energy,26,22.9858,29.7145,0.7951,-0.0859,-0.3723
5,Single: sensor_12_wlt_d1_energy,26,22.9931,29.7944,0.7940,-0.0786,-0.3405
6,Single: sensor_21_wlt_d1_energy,26,22.9960,29.7241,0.7950,-0.0757,-0.3281
7,Single: sensor_20_wlt_d1_std,26,22.9991,29.7440,0.7947,-0.0726,-0.3146
8,Baseline + d2_energy,32,22.9992,29.7398,0.7948,-0.0725,-0.3142
9,Single: sensor_15_wlt_d1_std,26,23.0037,29.7683,0.7944,-0.0679,-0.2944


,experiment,n_features,MAE,RMSE,R2,MAE_delta,MAE_change_%
0,Baseline + d2_std,32,22.9140,29.6756,0.7957,-0.1577,-0.6836
27,Baseline,25,23.0717,29.8586,0.7932,0.0000,0.0000
37,Baseline + ALL,60,23.3065,30.0047,0.7911,0.2348,1.0176


## Висновки

Validation (80/20 двигунів, XGBoost з тими самими параметрами):
- Baseline (25 ознак): MAE 23.0717, RMSE 29.8586, R2 0.7932.
- Baseline + ALL wavelet (35 ознак, 60 загалом): MAE 23.3065 (+0.2348, +1.02%), тобто гірше за baseline.
- **Baseline + d2_std** (7 ознак, фінальний набір): MAE 22.9140 (−0.1577, −0.68%), RMSE 29.6756, R2 0.7957.
  Найкращий груповий результат. Серед інших груп тільки `d2_energy` дає покращення (22.9992, −0.0725).
  `a2_mean` погіршує на 0.4260 (23.4977), `d1_energy` на 0.1233, `d1_std` на 0.0515.
- Найкращі одиночні ознаки: `sensor_12_wlt_a2_mean` 22.9368 (−0.1348), `sensor_11_wlt_d2_std` 22.9624 (−0.1092).
  Усі top-10 одиночних покращують baseline на 0.07–0.13.
- ALL без `d2_std` дає 23.5024 (+0.4307), тобто ця група найважливіша у повному наборі, але сам по собі ALL гірший за baseline.
